# Self-Contained Kaggle GPU Framework Validation

This notebook is self-contained. It does **not** clone from GitHub.

It will:
1. Reconstruct the needed project files from an embedded payload
2. Install a P100-compatible PyTorch build and the required dependencies
3. Run the GPU-scale validation
4. Bundle all outputs into one zip file
5. Print a compact summary in the notebook


In [ ]:
from pathlib import Path
import base64
import io
import os
import shutil
from zipfile import ZipFile

REPO_DIR = Path('/kaggle/working/EECE_608')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
REPO_DIR.mkdir(parents=True, exist_ok=True)

PAYLOAD_B64 = ''.join([
    '...'
    'AC8AXhAAAO8QAQAAAA=='
])
payload_bytes = base64.b64decode(PAYLOAD_B64.encode('ascii'))
with ZipFile(io.BytesIO(payload_bytes), 'r') as zf:
    zf.extractall(REPO_DIR)

os.chdir(REPO_DIR)
print('repo_dir =', REPO_DIR)
print('embedded_files =', len([p for p in REPO_DIR.rglob('*') if p.is_file()]))


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torch', 'torchvision', 'torchaudio'], check=False)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--force-reinstall',
    'torch==2.2.2', 'torchvision==0.17.2', 'torchaudio==2.2.2',
    '--index-url', 'https://download.pytorch.org/whl/cu121'
], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir',
    'numpy>=1.26', 'pandas>=2.2', 'PyYAML>=6.0', 'matplotlib>=3.8',
    'opacus>=1.5', 'dp-accounting>=0.4', 'scikit-learn>=1.4'
], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR), '--no-deps'], check=True)
print('install complete')


In [ ]:
import torch

print('torch_version =', torch.__version__)
print('cuda_available =', torch.cuda.is_available())
print('device_count =', torch.cuda.device_count())
print('arch_list =', torch.cuda.get_arch_list() if torch.cuda.is_available() else [])
if torch.cuda.is_available():
    print('device_name =', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('CUDA is not available. Enable GPU in the Kaggle notebook settings before running.')


In [ ]:
import subprocess
import sys

script_path = REPO_DIR / 'codex' / 'run_framework_validation_gpu_scale.py'
cmd = [sys.executable, str(script_path)]
print('running =', ' '.join(cmd))
subprocess.run(cmd, cwd=str(REPO_DIR), check=True)


In [ ]:
results_dir = REPO_DIR / 'codex' / 'results' / 'framework_validation_gpu_scale'
expected = [
    results_dir / 'framework_validation_gpu_scale_summary.json',
    results_dir / 'framework_validation_gpu_scale_rows.csv',
    results_dir / 'framework_validation_gpu_scale_checks.json',
    results_dir / 'framework_validation_gpu_scale_report.md',
    results_dir / 'framework_validation_gpu_scale_artifacts.zip',
]
for path in expected:
    print(path.name, 'exists =', path.exists())
print('bundled_download =', results_dir / 'framework_validation_gpu_scale_artifacts.zip')


In [ ]:
import json

summary_path = REPO_DIR / 'codex' / 'results' / 'framework_validation_gpu_scale' / 'framework_validation_gpu_scale_summary.json'
checks_path = REPO_DIR / 'codex' / 'results' / 'framework_validation_gpu_scale' / 'framework_validation_gpu_scale_checks.json'
rows_path = REPO_DIR / 'codex' / 'results' / 'framework_validation_gpu_scale' / 'framework_validation_gpu_scale_rows.csv'

summary = json.loads(summary_path.read_text(encoding='utf-8'))
checks = json.loads(checks_path.read_text(encoding='utf-8'))
audit_rows = summary.get('audit_rows', [])
failed_checks = [check for check in checks if check.get('status') == 'fail']
provisional = [row for row in audit_rows if row.get('result_trust') == 'provisional']
invalidated = [row for row in audit_rows if row.get('result_trust') == 'invalidated']

print('=== Validation Summary ===')
print('training_rows =', len(summary.get('training_rows', [])))
print('audit_rows =', len(audit_rows))
print('failed_checks =', len(failed_checks))
print('provisional_rows =', len(provisional))
print('invalidated_rows =', len(invalidated))
print('rows_csv =', rows_path)

if provisional:
    ranked = sorted(provisional, key=lambda row: -(row.get('epsilon_lower_conservative') or 0.0))
    print('\n=== Best Provisional Rows ===')
    for row in ranked[:5]:
        print(
            f"{row['dataset']} + {row['attack_name']}: eps_lower={row.get('epsilon_lower_conservative')}, eps_upper={row.get('epsilon_upper_tighter')}, trust={row.get('result_trust')}"
        )

if invalidated:
    print('\n=== Invalidated Rows ===')
    for row in invalidated:
        print(f"{row['dataset']} + {row['attack_name']}: tags={row.get('diagnostic_tags')}")

if failed_checks:
    print('\n=== Failed Checks ===')
    for check in failed_checks[:10]:
        print(f"{check['check_id']}: {check['details']}")
